# IDR + SIGReg planning results

The exact controls are the $H=1$ rows from the history ablation: IDR uses the same $\lambda=10$, and SIGReg uses the same weight 0.09.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

EXPERIMENT_DIR = Path.cwd().resolve()
if not (EXPERIMENT_DIR / "aggregated_results.csv").exists():
    candidate = Path("planning/experiments/idr_sigreg").resolve()
    if candidate.exists():
        EXPERIMENT_DIR = candidate

PLANNING_ROOT = next(
    candidate
    for candidate in [EXPERIMENT_DIR, *EXPERIMENT_DIR.parents]
    if (candidate / "plot_style.py").exists() and (candidate / "experiments").is_dir()
)
sys.path.insert(0, str(PLANNING_ROOT))
from plot_style import FULL_WIDTH, apply_matplotlib_style, palette

apply_matplotlib_style()
ENV_ORDER = ["TwoRoom", "Reacher", "Push-T", "OGBench-Cube"]
METHOD_ORDER = ["IDR", "SIGReg", "IDR + SIGReg"]
METHOD_COLORS = {
    "IDR": palette["Dark Blue"],
    "SIGReg": palette["Dark Red"],
    "IDR + SIGReg": palette["Med Purple"],
}


In [ ]:
combined = pd.read_csv(EXPERIMENT_DIR / "aggregated_results.csv")
combined = combined.query("status == 'ok'").copy()
assert len(combined) == 20, f"Expected 20 completed combined runs, found {len(combined)}"
assert combined.groupby("env_label").size().eq(5).all()
combined["display_method"] = "IDR + SIGReg"

history_path = EXPERIMENT_DIR.parent / "history_ablation" / "aggregated_results.csv"
history = pd.read_csv(history_path)
controls = history.query("status == 'ok' and history == 1 and method in ['idr', 'sigreg']").copy()
assert len(controls) == 40, f"Expected 40 exact-control runs, found {len(controls)}"
assert controls.groupby(["env_label", "method"]).size().eq(5).all()
controls["display_method"] = controls["method"].map({"idr": "IDR", "sigreg": "SIGReg"})

columns = ["env_label", "display_method", "seed", "success_rate"]
comparison = pd.concat([controls[columns], combined[columns]], ignore_index=True)
comparison.to_csv(EXPERIMENT_DIR / "comparison_results.csv", index=False)


In [ ]:
table = (
    comparison.groupby(["env_label", "display_method"], as_index=False)
    .agg(mean=("success_rate", "mean"), sem=("success_rate", "sem"), n=("success_rate", "size"))
)
table["env_label"] = pd.Categorical(table["env_label"], ENV_ORDER, ordered=True)
table["display_method"] = pd.Categorical(table["display_method"], METHOD_ORDER, ordered=True)
table = table.sort_values(["env_label", "display_method"]).reset_index(drop=True)
table.to_csv(EXPERIMENT_DIR / "comparison_summary_results.csv", index=False)
table

In [ ]:
fig, ax = plt.subplots(figsize=(FULL_WIDTH, 2.6))
x = np.arange(len(ENV_ORDER))
width = 0.24

for index, method in enumerate(METHOD_ORDER):
    method_rows = table[table["display_method"] == method].set_index("env_label").reindex(ENV_ORDER)
    ax.bar(
        x + (index - 1) * width,
        method_rows["mean"],
        width,
        yerr=method_rows["sem"],
        capsize=2,
        label=method,
        color=METHOD_COLORS[method],
    )

ax.set_xticks(x, ENV_ORDER)
ax.set_ylabel("Planning success rate (%)")
ax.set_ylim(0, 105)
ax.grid(axis="y")
ax.legend(ncol=3, loc="upper center", bbox_to_anchor=(0.5, 1.18))
fig.tight_layout()
fig.savefig(EXPERIMENT_DIR / "idr_sigreg_comparison.pdf")
plt.show()
